# Conditional dependence between source observations

A controlled two-source study compares independence, correct joint-sequential and composite-source representations. It evaluates physical risk, policies and certificates under a fixed family of 300 complete joint laws. This is a finite numerical study, not a natural-backend or LLM deployment experiment.

The default `RUN_PROFILE="pilot"` uses 100 calibration repetitions per truth/sample-size pair and three timing repetitions. The final profile uses 1,000 calibration repetitions and seven timing repetitions. Checkpoints are tied to the configuration fingerprint; `RUN_LABEL` selects a separate output directory for a repeated run. Dependencies are NumPy, pandas, SciPy and matplotlib. No external dataset or API is used.


In [ ]:
# Run configuration
RUN_PROFILE = "pilot"                 # "pilot" or "final"
OUTPUT_BASE = None                     # None: under the kernel's current working directory
RUN_LABEL = "default"                 # optional label for a separate repeat
RESUME = True

print("Loading local scientific libraries; no API or network call is used.")
import sys
import os
import json
import math
import time
import hashlib
import platform
import itertools
import traceback
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache

try:
    import numpy as np
    import pandas as pd
    import scipy
    from scipy.optimize import linprog
    from scipy.stats import beta, norm
    import matplotlib
    import matplotlib.pyplot as plt
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "This notebook needs numpy, pandas, scipy and matplotlib in the SAME "
        "Anaconda/Jupyter kernel. Install missing packages in that environment, "
        "restart the kernel, then Run All. It does not need OpenRouter."
    ) from exc

NOTEBOOK_VERSION = "issue8_joint_dependence_v1.0"
IMPLEMENTATION_SHA256 = "d60b4c03f89b3afe11fef0545900fb638485a829cae1fe64bb9fcd3e68fce40b"
if RUN_PROFILE not in ("pilot", "final"):
    raise ValueError('RUN_PROFILE must be exactly "pilot" or "final".')
if not RUN_LABEL or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in RUN_LABEL):
    raise ValueError("RUN_LABEL may contain letters, digits, underscores and hyphens.")

PROFILE_R = {"A": (0.70, 0.90), "B": (0.80, 0.80), "C": (0.90, 0.70)}
DEPENDENCE_GRID = (-1.0, -0.5, 0.0, 0.125, 0.15, 0.25, 0.5, 0.75, 0.95, 1.0)
PRIOR_Y1 = 0.50
QUERY_COSTS = np.array([0.005, 0.005], dtype=float)
ABSTENTION_COST = 0.10
CALIBRATION_TRUTHS = ((0.0, 0.0), (0.25, 0.0), (0.75, 0.75), (1.0, 1.0))
CALIBRATION_N_PER_CLASS = (50, 200, 1000)
CALIBRATION_REPETITIONS = 100 if RUN_PROFILE == "pilot" else 1000
TIMING_REPETITIONS = 3 if RUN_PROFILE == "pilot" else 7
CALIBRATION_FAILURE_BUDGET = 0.05
MASTER_SEED = 20261010
PROB_TOL = 5e-13                 # validation only; never an epsilon probability floor
VALUE_TOL = 2e-9                # LP / hierarchy / certificate tolerance
TIE_TOL = 1e-12                 # choose the first action within this tolerance
REPS = ("independence", "joint_sequential", "composite")
METHODS = ("Nominal", "Static_randomized", "HCR", "Rectangular")
CELLS = ((0, 0), (0, 1), (1, 0), (1, 1))
LOSS = np.array([[ABSTENTION_COST, ABSTENTION_COST], [0.0, 1.0], [1.0, 0.0]])
ACTION_LABELS = ("abstain", "answer0", "answer1")
ROUNDING_AUDIT = []
RUN_READY = False

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def plain(value):
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    if isinstance(value, np.ndarray):
        return plain(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value

def canonical_json(value):
    return json.dumps(plain(value), sort_keys=True, separators=(",", ":"), allow_nan=False)

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(plain(value), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)

ENVIRONMENT = {
    "python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
    "platform": platform.platform(), "machine": platform.machine(),
    "processor": platform.processor(), "cpu_count": os.cpu_count(),
    "timing_clock": "time.perf_counter", "timing_is_wall_clock": True,
}
PROTOCOL = {
    "version": NOTEBOOK_VERSION, "implementation_sha256": IMPLEMENTATION_SHA256,
    "profile": RUN_PROFILE, "label": RUN_LABEL, "marginal_profiles": PROFILE_R,
    "dependence_grid": DEPENDENCE_GRID, "prior_y1": PRIOR_Y1,
    "query_costs": QUERY_COSTS.tolist(), "abstention_cost": ABSTENTION_COST,
    "calibration_truths_t1_t0": CALIBRATION_TRUTHS,
    "calibration_n_per_class": CALIBRATION_N_PER_CLASS,
    "calibration_repetitions": CALIBRATION_REPETITIONS,
    "calibration_failure_budget": CALIBRATION_FAILURE_BUDGET,
    "master_seed": MASTER_SEED, "timing_repetitions": TIMING_REPETITIONS,
    "value_tolerance": VALUE_TOL, "tie_tolerance": TIE_TOL,
    "calibration_candidates": "fixed 300-law joint family / 3 product projections",
    "static_policy_class": "all mixtures of complete deterministic trees",
    "nature_observes_current_action": True,
}
RUN_FINGERPRINT = hashlib.sha256(canonical_json({"protocol": PROTOCOL, "environment": ENVIRONMENT}).encode()).hexdigest()
base = Path.cwd() / "rose_issue8_outputs_v1" if OUTPUT_BASE is None else Path(OUTPUT_BASE).expanduser()
OUTPUT_DIR = (base / (RUN_PROFILE + "_" + RUN_LABEL + "_" + RUN_FINGERPRINT[:12])).resolve()
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "issue8_manifest.json"
if manifest_path.exists():
    previous_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if previous_manifest.get("fingerprint") != RUN_FINGERPRINT:
        raise RuntimeError("Checkpoint fingerprint differs. Use a new RUN_LABEL; do not mix runs.")
    MANIFEST = previous_manifest
else:
    MANIFEST = {"created_utc": utc_now(), "fingerprint": RUN_FINGERPRINT, "protocol": PROTOCOL, "environment": ENVIRONMENT}
    atomic_json(manifest_path, MANIFEST)

def save_frame(name, rows):
    frame = rows.copy() if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    target = OUTPUT_DIR / name
    tmp = target.with_name(target.name + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, target)
    return frame

def status(stage, state, **extra):
    atomic_json(OUTPUT_DIR / "issue8_run_status.json", {
        "fingerprint": RUN_FINGERPRINT, "profile": RUN_PROFILE,
        "stage": stage, "state": state, "updated_utc": utc_now(), **extra,
    })

def safely_run(stage, func):
    status(stage, "running")
    try:
        result = func()
    except Exception as exc:
        status(stage, "failed", error_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc())
        raise
    status(stage, "completed")
    return result

print("PROFILE:", RUN_PROFILE)
print("Calibration repetitions per truth/n:", CALIBRATION_REPETITIONS)
print("Fixed prior=0.50; lambda_A=0.10; costs=(0.005, 0.005).")
print("OUTPUT:", OUTPUT_DIR)
print("Pilot/final checkpoints are separated automatically.")


## Joint-law design and representations

The physical law is a complete conditional table $P_\theta(E_1,E_2\mid Y)$ with binary $Y,E_1,E_2$.
The class-conditional marginals are fixed by one profile:

| Profile | $r_1=q_{1,1}$ | $r_2=q_{2,1}$ |
|---|---:|---:|
| A | 0.70 | 0.90 |
| B | 0.80 | 0.80 |
| C | 0.90 | 0.70 |

For both sources, $q_{i,0}=1-r_i$. Within each class $y$, define

$$
\begin{aligned}
p_{11|y}&=q_{1y}q_{2y}+\delta_y,&p_{10|y}&=q_{1y}(1-q_{2y})-\delta_y,\\
p_{01|y}&=(1-q_{1y})q_{2y}-\delta_y,&p_{00|y}&=(1-q_{1y})(1-q_{2y})+\delta_y.
\end{aligned}
$$

The Fréchet endpoints are $\delta_y^- = \max(0,q_{1y}+q_{2y}-1)-q_{1y}q_{2y}$ and
$\delta_y^+=\min(q_{1y},q_{2y})-q_{1y}q_{2y}$.
For $t_y\ge0$, $\delta_y=t_y\delta_y^+$; for $t_y<0$, $\delta_y=(-t_y)\delta_y^-$.
The grid is $(-1,-.5,0,.125,.15,.25,.5,.75,.95,1)$ independently for **$(t_1,t_0)$**;
there are $3\times10\times10=300$ physical laws. Array order is **[Y, E1, E2]**, and flattened cells are 00, 01, 10, 11.
$t_y$ is normalized dependence, **not Pearson correlation**. The catalog reports
$\rho_y=\delta_y/\sqrt{q_{1y}(1-q_{1y})q_{2y}(1-q_{2y})}$ for each class separately.

All main experiments hold $P(Y=1)=.5$, $c_1=c_2=.005$, and $\lambda_A=.10$ fixed.
These are synthetic design assumptions, not empirical operational estimates.
The reference prior is the correct fixed prior in this experiment; Issue 6 is not re-tested here.

| Representation | Planning channel | Agent's observations | Charge |
|---|---|---|---|
| `independence` | product of the physical law's exact class marginals | sequential; early stopping allowed | .005 per acquired source |
| `joint_sequential` | correct complete joint law; history-conditioned remaining kernel | sequential; early stopping allowed | .005 per acquired source |
| `composite` | the same complete joint law | one four-valued vector $(E_1,E_2)$ | .010 for two acquired sources |

The first two share the same agent policy class. Composite changes both the observation/action class
and Nature's opportunities to reoptimize; no general robust-value ordering is asserted across representations.
The reports distinguish **logical query actions** from **original source acquisitions**.
Zero probabilities at Fréchet endpoints are constructed algebraically as **exact structural zeros**.
Both small positive and negative endpoint cancellation errors would otherwise change compatibility;
the endpoint construction logs its roundoff adjustment. Away from endpoints, only tiny negative roundoff
and normalization discrepancies are repaired and logged. There is no epsilon probability floor.


In [ ]:
def model_id(profile, t1, t0):
    return "{}|t1={:+.3f}|t0={:+.3f}".format(profile, float(t1), float(t0))

def checked_table(table, label):
    table = np.asarray(table, dtype=float).copy()
    if table.shape != (2, 2, 2) or not np.all(np.isfinite(table)):
        raise ValueError("Invalid channel table: " + str(label))
    minimum = float(table.min())
    if minimum < -PROB_TOL:
        raise ValueError("Negative joint probability in " + str(label))
    small_negative = (table < 0.0)
    if small_negative.any():
        ROUNDING_AUDIT.append({"model": label, "operation": "zero tiny negative roundoff", "minimum_before": minimum})
        table[small_negative] = 0.0
    for y in (0, 1):
        discrepancy = 1.0 - float(table[y].sum())
        if abs(discrepancy) > PROB_TOL:
            raise ValueError("Unnormalized conditional law: " + str(label))
        if discrepancy != 0.0:
            j = np.unravel_index(int(np.argmax(table[y])), (2, 2))
            table[(y,) + j] += discrepancy
            ROUNDING_AUDIT.append({"model": label, "y": y, "operation": "normalization roundoff assigned to largest cell", "amount": discrepancy})
    if np.any(table < 0.0):
        raise AssertionError("Roundoff repair produced a negative probability.")
    return table

def make_model(profile, t1, t0):
    r1, r2 = PROFILE_R[profile]
    identifier = model_id(profile, t1, t0)
    table = np.zeros((2, 2, 2), dtype=float)
    rows = []
    for y, t in ((0, float(t0)), (1, float(t1))):
        if not -1.0 <= t <= 1.0:
            raise ValueError("Normalized dependence must be in [-1,1].")
        q1, q2 = ((1.0-r1, 1.0-r2) if y == 0 else (r1, r2))
        lower = max(0.0, q1+q2-1.0) - q1*q2
        upper = min(q1, q2) - q1*q2
        delta = t*upper if t >= 0.0 else (-t)*lower
        table[y, 1, 1] = q1*q2 + delta
        table[y, 1, 0] = q1*(1.0-q2) - delta
        table[y, 0, 1] = (1.0-q1)*q2 - delta
        table[y, 0, 0] = (1.0-q1)*(1.0-q2) + delta
        # Construct Frechet endpoints algebraically, preserving STRUCTURAL zeros.
        # Even a tiny positive roundoff would otherwise make an impossible history
        # appear compatible and could change an HCR terminal ambiguity set.
        endpoint = None
        if t == 1.0:
            p11 = min(q1,q2)
            endpoint = np.array([[1.0-max(q1,q2),q2-p11],[q1-p11,p11]])
        elif t == -1.0:
            if q1+q2 <= 1.0:
                endpoint = np.array([[1.0-q1-q2,q2],[q1,0.0]])
            else:
                endpoint = np.array([[0.0,1.0-q1],[1.0-q2,q1+q2-1.0]])
        if endpoint is not None:
            adjustment = float(np.max(np.abs(endpoint-table[y])))
            if adjustment > PROB_TOL:
                raise AssertionError("Endpoint construction differs beyond roundoff.")
            if adjustment != 0.0:
                ROUNDING_AUDIT.append({"model":identifier,"y":y,
                    "operation":"algebraic Frechet endpoint; preserve exact structural zeros",
                    "amount":adjustment})
            table[y] = endpoint
        rho = delta / math.sqrt(q1*(1.0-q1)*q2*(1.0-q2))
        rows.append({"y": y, "t": t, "q1": q1, "q2": q2, "delta": delta,
                     "delta_lower": lower, "delta_upper": upper, "rho_conditional": rho})
    table = checked_table(table, identifier)
    for row in rows:
        y = row["y"]
        if abs(float(table[y, 1, :].sum()) - row["q1"]) > PROB_TOL:
            raise AssertionError("First marginal changed.")
        if abs(float(table[y, :, 1].sum()) - row["q2"]) > PROB_TOL:
            raise AssertionError("Second marginal changed.")
        if abs(row["rho_conditional"]) > 1.0 + PROB_TOL:
            raise AssertionError("Invalid conditional correlation.")
    return {"id": identifier, "profile": profile, "t1": float(t1), "t0": float(t0), "table": table, "parameters": rows}

def product_projection(model):
    table = np.zeros((2, 2, 2), dtype=float)
    for y in (0, 1):
        q1 = float(model["table"][y, 1, :].sum())
        q2 = float(model["table"][y, :, 1].sum())
        table[y] = np.outer([1.0-q1, q1], [1.0-q2, q2])
    return {"id": "product|" + model["profile"], "profile": model["profile"],
            "t1": 0.0, "t0": 0.0, "table": checked_table(table, "product|"+model["profile"])}

def same_law(a, b):
    return bool(np.allclose(a["table"], b["table"], atol=PROB_TOL, rtol=0.0))

def unique_models(models):
    unique = []
    for model in models:
        if not any(same_law(model, previous) for previous in unique):
            unique.append(model)
    return unique

def is_represented(truth, models):
    return any(same_law(truth, model) for model in models)

def marginal_profile_represented(truth, models):
    return is_represented(product_projection(truth), unique_models([product_projection(m) for m in models]))

def marginal_likelihood(tables, history):
    mask = np.ones((2, 2), dtype=bool)
    for source, observation in history:
        if source == 0:
            mask &= np.array([[observation == 0]*2, [observation == 1]*2])
        elif source == 1:
            mask &= np.array([[observation == 0, observation == 1]]*2)
        else:
            raise ValueError("Sequential source index must be 0 or 1.")
    return np.sum(tables[:, :, mask], axis=2)

def history_text(history):
    return "root" if not history else ";".join("E{}={}".format(i+1, e) for i, e in history)

def posterior_and_predictive(tables, history, source=None):
    likelihood = marginal_likelihood(tables, history)
    weights = likelihood * np.array([1.0-PRIOR_Y1, PRIOR_Y1])[None, :]
    normalizers = weights.sum(axis=1)
    active = normalizers > 0.0
    if not active.any():
        return active, np.empty((0, 2)), None, None
    b = weights[active] / normalizers[active, None]
    if source is None:
        return active, b, None, None
    if source in [i for i, _ in history]:
        raise ValueError("A source may be queried only once.")
    cond = np.zeros((int(active.sum()), 2, 2))
    for e in (0, 1):
        numerator = marginal_likelihood(tables, history + ((source, e),))[active]
        denominator = likelihood[active]
        # A zero conditional row is irrelevant when the associated posterior mass is zero.
        # The two-source protocol has strictly positive class likelihoods at first-step histories.
        if np.any(denominator <= 0.0):
            raise AssertionError("Undefined conditional row outside the frozen two-source protocol.")
        cond[:, :, e] = numerator / denominator
    if not np.allclose(cond.sum(axis=2), 1.0, atol=PROB_TOL, rtol=0.0):
        raise AssertionError("Conditional kernels are not normalized.")
    pred = np.einsum("ky,kye->ke", b, cond)
    return active, b, cond, pred

def robust_stop(beliefs):
    risks = np.max(beliefs @ LOSS.T, axis=0)
    action = int(np.flatnonzero(risks <= risks.min() + TIE_TOL)[0])
    return float(risks[action]), action, risks

CATALOG = [make_model(profile, t1, t0) for profile in PROFILE_R for t1 in DEPENDENCE_GRID for t0 in DEPENDENCE_GRID]
CATALOG_BY_ID = {m["id"]: m for m in CATALOG}
CATALOG_INDEX = {m["id"]: i for i, m in enumerate(CATALOG)}
PRODUCT_CANDIDATES = [product_projection(make_model(profile, 0.0, 0.0)) for profile in PROFILE_R]
CATALOG_TABLES = np.stack([m["table"] for m in CATALOG])
PRODUCT_TABLES = np.stack([m["table"] for m in PRODUCT_CANDIDATES])

catalog_rows = []
for model in CATALOG:
    for row in model["parameters"]:
        y = row["y"]
        catalog_rows.append({"model_id": model["id"], "profile": model["profile"], "t1": model["t1"], "t0": model["t0"], **row,
            **{"p{}{}".format(a,b): float(model["table"][y,a,b]) for a,b in CELLS}})
save_frame("issue8_distribution_catalog.csv", catalog_rows)
print("Joint candidates:", len(CATALOG), "; unique product candidates:", len(PRODUCT_CANDIDATES))
print("Joint table axis order: [Y, E1, E2]; truth parameters are always (t1, t0).")


## Exact policy evaluation and randomized Static

For two binary sources used at most once, there are 291 complete deterministic sequential trees:
$N_0=3$, $N_1=3+N_0^2=12$, $N_2=3+2N_1^2=291$.
The composite class contains $3+3^4=84$ deterministic trees.
All true-law metrics are finite sums over the **physical joint distribution**, including under independence planning.
Reported wrong-answer and abstention probabilities are unconditional probabilities over tasks.

Static minimizes $v$ over the convex hull of the full deterministic policy bank:

$$\min_{w\ge0,\;\sum_\pi w_\pi=1}v
\quad\text{s.t.}\quad \sum_\pi w_\pi R_\theta(\pi)\le v\quad \forall\theta.$$

This is the randomized Static optimum, not merely the best deterministic tree.
The LP uses SciPy/HiGHS and checks primal feasibility, dual feasibility, stationarity, complementarity and the gap.
Every positive LP weight is retained. Mixture metrics are evaluated analytically rather than sampled.
Tie handling is fixed: abstain, answer0, answer1, then source1 and source2 within $10^{-12}$ at decision nodes.
All four formulations coincide in objective on singleton libraries; that comparison diagnoses dependence
misspecification rather than a special HCR benefit.


In [ ]:
# A terminal action is encoded by 0=abstain, 1=answer0, 2=answer1.
# Sequential query node: ("q", source, child0, child1).
# Composite query node: ("v", child00, child01, child10, child11).
@lru_cache(maxsize=None)
def enumerate_sequential(remaining):
    policies = [0, 1, 2]
    for source in remaining:
        children = enumerate_sequential(tuple(i for i in remaining if i != source))
        policies.extend(("q", source, a, b) for a in children for b in children)
    return tuple(policies)

SEQUENTIAL_POLICIES = enumerate_sequential((0, 1))
COMPOSITE_POLICIES = tuple([0, 1, 2] + [("v",)+children for children in itertools.product((0,1,2), repeat=4)])

def normalize_policy(policy):
    return tuple(normalize_policy(p) for p in policy) if isinstance(policy, (list, tuple)) else int(policy) if isinstance(policy, (int, np.integer)) else policy

def policy_json(policy):
    if isinstance(policy, (int, np.integer)):
        return {"action": ACTION_LABELS[int(policy)]}
    if policy[0] == "q":
        return {"action": "query", "source": int(policy[1])+1, "children": {"0": policy_json(policy[2]), "1": policy_json(policy[3])}}
    return {"action": "query_composite", "original_sources": [1,2], "children": {
        "{}{}".format(*cell): policy_json(policy[1+j]) for j,cell in enumerate(CELLS)}}

def policy_id(policy):
    return hashlib.sha256(canonical_json(policy).encode()).hexdigest()[:20]

def policies_for(rep):
    return COMPOSITE_POLICIES if rep == "composite" else SEQUENTIAL_POLICIES

def traverse(policy, cell):
    queried = np.zeros(2, dtype=float)
    logical = 0
    history = ()
    while not isinstance(policy, (int, np.integer)):
        if policy[0] == "q":
            source = int(policy[1])
            if queried[source] != 0:
                raise AssertionError("Repeated source query.")
            observation = cell[source]
            queried[source] = 1.0
            logical += 1
            history += ((source, observation),)
            policy = policy[2+observation]
        else:
            queried[:] = 1.0
            logical += 1
            history = ((0, cell[0]), (1, cell[1]))
            policy = policy[1+CELLS.index(cell)]
    return int(policy), queried, logical, history

def compile_policy_bank(policies):
    terminal = np.zeros((len(policies), 4), dtype=int)
    queried = np.zeros((len(policies), 4, 2), dtype=float)
    logical = np.zeros((len(policies), 4), dtype=float)
    for j, policy in enumerate(policies):
        for c, cell in enumerate(CELLS):
            terminal[j,c], queried[j,c], logical[j,c], _ = traverse(policy, cell)
    wrong = np.zeros((len(policies), 2, 4), dtype=float)
    wrong[:,0,:] = terminal == 2
    wrong[:,1,:] = terminal == 1
    abstain = (terminal == 0).astype(float)
    return {"terminal": terminal, "queried": queried, "logical": logical, "wrong": wrong, "abstain": abstain}

BANKS = {"sequential": compile_policy_bank(SEQUENTIAL_POLICIES), "composite": compile_policy_bank(COMPOSITE_POLICIES)}
POLICY_INDEX = {
    "sequential": {canonical_json(p): i for i,p in enumerate(SEQUENTIAL_POLICIES)},
    "composite": {canonical_json(p): i for i,p in enumerate(COMPOSITE_POLICIES)},
}

def all_policy_metrics(tables, rep):
    tables = np.asarray(tables, dtype=float)
    bank = BANKS["composite" if rep == "composite" else "sequential"]
    joint = tables.reshape(len(tables), 2, 4) * np.array([1.0-PRIOR_Y1, PRIOR_Y1])[None,:,None]
    cell_mass = joint.sum(axis=1)
    wrong = np.einsum("kyc,pyc->kp", joint, bank["wrong"])
    abstain = cell_mass @ bank["abstain"].T
    q0 = cell_mass @ bank["queried"][:,:,0].T
    q1 = cell_mass @ bank["queried"][:,:,1].T
    logical = cell_mass @ bank["logical"].T
    cost = QUERY_COSTS[0]*q0 + QUERY_COSTS[1]*q1
    return {"wrong": wrong, "abstain": abstain, "correct": 1.0-wrong-abstain,
            "source1_queries": q0, "source2_queries": q1, "original_queries": q0+q1,
            "logical_queries": logical, "query_cost": cost,
            "risk": wrong + ABSTENTION_COST*abstain + cost}

def mixture_metrics(models, rep, solution):
    metrics = all_policy_metrics(np.stack([m["table"] for m in models]), rep)
    support = np.asarray(solution["support_indices"], dtype=int)
    weights = np.asarray(solution["support_weights"], dtype=float)
    return {key: value[:,support] @ weights for key,value in metrics.items()}

def deterministic_solution(policy, objective, rep, method, **extra):
    idx = POLICY_INDEX["composite" if rep == "composite" else "sequential"][canonical_json(policy)]
    return {"method": method, "representation": rep, "objective": float(objective),
            "support_indices": [idx], "support_weights": [1.0], **extra}

def solution_signature(solution):
    policies = policies_for(solution["representation"])
    support = sorted((policy_id(policies[i]), round(float(w),12)) for i,w in zip(solution["support_indices"],solution["support_weights"]))
    return hashlib.sha256(canonical_json(support).encode()).hexdigest()[:20]

def argmin_tie(values):
    values = np.asarray(values, dtype=float)
    return int(np.flatnonzero(values <= np.min(values) + TIE_TOL)[0])

print("Complete deterministic policy counts:", len(SEQUENTIAL_POLICIES), "sequential;", len(COMPOSITE_POLICIES), "composite.")


## History-dependent HCR and the explicitly specified Rectangular outer approximation

For the complete observation history $h$,

$$L_\theta(h\mid y)=P_\theta(H_h\mid Y=y),\qquad
b_h^\theta(y)=\frac{b_0(y)L_\theta(h\mid y)}{\sum_z b_0(z)L_\theta(h\mid z)},$$

$$Q_{i,h}^\theta(e\mid y)=\frac{P_\theta(E_i=e,H_h\mid Y=y)}{P_\theta(H_h\mid Y=y)}.$$

HCR uses the posterior and remaining-source kernel from the **same complete model** at the current history.
Its continuation value is

$$C_i(h)=c_i+\max_{\theta:L_\theta(h)>0}\sum_e P_\theta(e\mid h)V(h,i,e).$$

Each successor is reoptimized over all complete models compatible with that extended history.
Using only $b_h$, the model label and remaining indices is insufficient for arbitrary dependence unless
the history or residual conditional law is retained. The XOR analytic control demonstrates this explicitly.

The Rectangular comparison here is a defined **outer approximation**, not a claim that all possible
rectangular constructions are identical. At the root it uses $b_0$ and the library's first-source kernels.
After first observation $h=(i,e)$ it may independently combine $b_h^{\theta_A}$ with
$Q_{j,h}^{\theta_B}$ for the remaining source. Its final belief set contains all positive-normalizer
Bayes updates of those spliced pairs. The remaining kernels still condition on the **same observed history**;
they are not replaced with unconditional marginals. With two sources, all class likelihoods at first-step
histories are strictly positive, so these conditional rows are defined even at Fréchet endpoints.
Zero-probability complete histories are skipped, not floored.

The code checks, **within each representation and the same policy class**,
$V_{\rm Nominal}\le V_{\rm Static(randomized)}\le V_{\rm HCR}\le V_{\rm Rectangular}$.
This is an ordering of modeled robust values, not a universal ordering of physical risks.
The one-step composite HCR and Rectangular values coincide because its prior belief set is a singleton
and no intermediate query remains. Static can still differ from them.


In [ ]:
class HistoryPlanner:
    """Finite two-source HCR or specified joint rectangular outer approximation."""
    def __init__(self, models, representation="joint_sequential", rectangular=False):
        if not models:
            raise ValueError("An empty ambiguity set is not a robust planning problem.")
        self.models = models
        self.tables = np.stack([m["table"] for m in models])
        self.rep = representation
        self.rectangular = bool(rectangular)
        self.memo = {}
        self.state_data = {}

    def beliefs(self, history):
        if self.rectangular and self.rep != "composite" and len(history) == 2:
            first = history[:1]
            source, observation = history[1]
            _, b, kernel, _ = posterior_and_predictive(self.tables, first, source)
            # Both factors refer to the SAME observed first-step history,
            # but their model labels may differ under Rectangular.
            weights = b[:,None,:] * kernel[None,:,:,observation]
            normalizers = weights.sum(axis=2)
            positive = normalizers > 0.0
            return weights[positive] / normalizers[positive][:,None]
        _, b, _, _ = posterior_and_predictive(self.tables, history)
        return b

    def predictive(self, history, source):
        _, b, kernel, pred = posterior_and_predictive(self.tables, history, source)
        if self.rectangular and len(history) == 1:
            # K posterior choices x K history-conditioned kernel choices.
            return np.einsum("ay,bye->abe", b, kernel).reshape(-1,2)
        return pred

    def composite_predictive(self):
        return np.einsum("y,kyab->kab", np.array([1.0-PRIOR_Y1, PRIOR_Y1]), self.tables).reshape(len(self.models),4)

    def solve_state(self, history=()):
        if history in self.memo:
            return self.memo[history]
        b = self.beliefs(history)
        if len(b) == 0:
            self.memo[history] = (0.0, 0)
            self.state_data[history] = {"active": False}
            return self.memo[history]
        stop_value, stop_action, stop_losses = robust_stop(b)
        candidates = [(stop_value, stop_action)]
        query_values = {}
        if self.rep == "composite" and not history:
            successors = tuple(((0,a),(1,c)) for a,c in CELLS)
            solved = [self.solve_state(h) for h in successors]
            vector = np.array([value for value,_ in solved])
            value = float(QUERY_COSTS.sum() + np.max(self.composite_predictive() @ vector))
            query_values["composite"] = value
            candidates.append((value, ("v",)+tuple(p for _,p in solved)))
        elif self.rep != "composite":
            for source in (0,1):
                if source in [i for i,_ in history]:
                    continue
                solved = [self.solve_state(history+((source,e),)) for e in (0,1)]
                value = float(QUERY_COSTS[source] + np.max(self.predictive(history,source) @ np.array([v for v,_ in solved])))
                query_values[str(source)] = value
                candidates.append((value, ("q",source,solved[0][1],solved[1][1])))
        chosen = argmin_tie([value for value,_ in candidates])
        self.memo[history] = candidates[chosen]
        self.state_data[history] = {"active": True, "stop_value": stop_value,
            "stop_action": stop_action, "stop_losses": stop_losses.tolist(),
            "query_values": query_values, "posterior_y1_min": float(b[:,1].min()),
            "posterior_y1_max": float(b[:,1].max()), "belief_entries": len(b)}
        return self.memo[history]

    def evaluate_policy(self, policy, history=()):
        b = self.beliefs(history)
        if len(b) == 0:
            return 0.0
        if isinstance(policy, (int, np.integer)):
            return float(np.max(b @ LOSS[int(policy)]))
        if policy[0] == "v":
            child_values = [self.evaluate_policy(policy[1+j], ((0,a),(1,c))) for j,(a,c) in enumerate(CELLS)]
            return float(QUERY_COSTS.sum() + np.max(self.composite_predictive() @ child_values))
        source = int(policy[1])
        values = [self.evaluate_policy(policy[2+e], history+((source,e),)) for e in (0,1)]
        return float(QUERY_COSTS[source] + np.max(self.predictive(history,source) @ values))

def solve_history(models, rep, rectangular=False, return_planner=False):
    planner = HistoryPlanner(models, rep, rectangular)
    value, policy = planner.solve_state()
    checked = planner.evaluate_policy(policy)
    if abs(value-checked) > VALUE_TOL:
        raise AssertionError("History planner failed its prescribed-policy recursion check.")
    solution = deterministic_solution(policy, value, rep, "Rectangular" if rectangular else "HCR",
        objective_recomputed=checked, dynamic_state_count=len(planner.memo),
        table_array_bytes=int(planner.tables.nbytes))
    return (solution, planner) if return_planner else solution

def solve_nominal(models, nominal_index, rep):
    metrics = all_policy_metrics(np.stack([models[nominal_index]["table"]]), rep)
    index = argmin_tie(metrics["risk"][0])
    return deterministic_solution(policies_for(rep)[index],metrics["risk"][0,index],rep,"Nominal",
        nominal_model_id=models[nominal_index]["id"], dynamic_state_count=0)

def solve_static(models, rep):
    risk = all_policy_metrics(np.stack([m["table"] for m in models]), rep)["risk"]
    if len(models) == 1:
        index = argmin_tie(risk[0])
        return deterministic_solution(policies_for(rep)[index],risk[0,index],rep,"Static_randomized",
            lp_status="singleton_closed_form", lp_primal_residual=0.0,
            lp_dual_value=float(risk[0,index]), lp_duality_gap=0.0,
            lp_stationarity_residual=0.0, lp_complementarity_residual=0.0,
            static_deterministic_objective=float(risk[0,index]), randomization_gain=0.0)
    n_policies = risk.shape[1]
    # Variables: all deterministic-policy weights, followed by free epigraph v.
    objective = np.zeros(n_policies+1)
    objective[-1] = 1.0
    aub = np.column_stack([risk, -np.ones(len(models))])
    aeq = np.concatenate([np.ones(n_policies), [0.0]])[None,:]
    result = linprog(objective, A_ub=aub, b_ub=np.zeros(len(models)),
        A_eq=aeq, b_eq=np.array([1.0]),
        bounds=[(0.0,None)]*n_policies + [(None,None)], method="highs",
        options={"primal_feasibility_tolerance": 1e-9, "dual_feasibility_tolerance": 1e-9,
                 "ipm_optimality_tolerance":1e-10})
    if not result.success or result.x is None:
        raise RuntimeError("Static LP failed: " + str(result.message))
    weights = np.asarray(result.x[:-1], dtype=float)
    if weights.min() < -VALUE_TOL:
        raise AssertionError("Static LP returned a materially negative policy weight.")
    tiny_negative = weights < 0.0
    repair_mass = float(-weights[tiny_negative].sum())
    weights[tiny_negative] = 0.0
    total = float(weights.sum())
    if abs(total-1.0) > VALUE_TOL:
        raise AssertionError("Static policy weights do not sum to one.")
    weights /= total
    # Retain EVERY positive weight; no support truncation that changes the policy.
    support = np.flatnonzero(weights > 0.0)
    implemented_objective = float(np.max(risk @ weights))
    lp_value = float(result.fun)
    primal_residual = max(float(np.maximum(aub @ result.x,0.0).max()),
        float(np.abs(aeq @ result.x - 1.0).max()), float(max(0.0,-result.x[:-1].min())),
        abs(implemented_objective-lp_value))
    dual_ub = np.asarray(result.ineqlin.marginals)
    dual_eq = np.asarray(result.eqlin.marginals)
    dual_lower = np.asarray(result.lower.marginals)
    dual_upper = np.asarray(result.upper.marginals)
    stationarity = objective - aub.T @ dual_ub - aeq.T @ dual_eq - dual_lower - dual_upper
    dual_value = float(dual_eq[0])  # all inequality RHS and finite lower bounds are zero
    dual_feasibility = max(float(np.maximum(dual_ub,0.0).max()),
        float(np.maximum(-dual_lower[:-1],0.0).max()),abs(float(dual_ub.sum())+1.0),
        abs(float(dual_lower[-1])),float(np.abs(dual_upper).max()))
    complementarity = max(float(np.abs(np.asarray(result.ineqlin.residual)*dual_ub).max()),
        float(np.abs(result.x[:-1]*dual_lower[:-1]).max()))
    dual_gap = max(abs(lp_value-dual_value), abs(implemented_objective-dual_value))
    if max(primal_residual, dual_gap, dual_feasibility, float(np.abs(stationarity).max()), complementarity) > VALUE_TOL:
        raise AssertionError("Static LP primal/dual validation failed.")
    deterministic = float(np.min(np.max(risk,axis=0)))
    return {"method":"Static_randomized", "representation":rep,
        "objective":implemented_objective, "lp_objective":lp_value,
        "support_indices":support.tolist(), "support_weights":weights[support].tolist(),
        "lp_status":int(result.status), "lp_message":str(result.message),
        "lp_primal_residual":primal_residual, "lp_dual_value":dual_value,
        "lp_duality_gap":dual_gap, "lp_stationarity_residual":float(np.abs(stationarity).max()),
        "lp_complementarity_residual":complementarity, "lp_negative_mass_repaired":repair_mass,
        "lp_dual_feasibility_residual":dual_feasibility,
        "lp_adversarial_model_weights":(-dual_ub).tolist(),
        "static_deterministic_objective":deterministic,
        "randomization_gain":deterministic-implemented_objective}

def solve_methods(models, nominal_index, rep):
    nominal = solve_nominal(models,nominal_index,rep)
    static = solve_static(models,rep)
    hcr, hp = solve_history(models,rep,False,True)
    rect, rp = solve_history(models,rep,True,True)
    solutions = {s["method"]:s for s in (nominal,static,hcr,rect)}
    ordered = [solutions[m]["objective"] for m in METHODS]
    if any(ordered[j] > ordered[j+1]+VALUE_TOL for j in range(3)):
        raise AssertionError("Nominal <= randomized Static <= HCR <= Rectangular failed within " + rep)
    if rep == "composite" and abs(hcr["objective"]-rect["objective"]) > VALUE_TOL:
        raise AssertionError("One-step composite HCR and Rectangular should coincide.")
    if len(models) == 1 and max(ordered)-min(ordered) > VALUE_TOL:
        raise AssertionError("Singleton solvers should coincide.")
    return solutions, hp, rp


In [ ]:
def representation_models(physical, nominal, rep):
    if rep == "independence":
        planning = unique_models([product_projection(m) for m in physical])
        target = product_projection(nominal)
    else:
        planning = unique_models(physical)
        target = nominal
    matches = [j for j,m in enumerate(planning) if same_law(m,target)]
    if len(matches) != 1:
        raise AssertionError("Nominal model was lost or duplicated during projection/deduplication.")
    return planning, matches[0]

def robust_library(s, family):
    dependence = [(s,s)] if family == "matched_size_K3" else [(0.0,0.0),(s,s),(s,0.0),(0.0,s)]
    physical = unique_models([CATALOG_BY_ID[model_id(profile,t1,t0)] for profile in PROFILE_R for t1,t0 in dependence])
    nominal = CATALOG_BY_ID[model_id("B",s,s)]
    if not is_represented(nominal,physical):
        raise AssertionError("Robust library does not contain its nominal model.")
    return physical, nominal

def on_policy_history(policy, target):
    if not target:
        return True
    history = ()
    cursor = policy
    for i,e in target:
        if isinstance(cursor,(int,np.integer)) or cursor[0] != "q" or cursor[1] != i:
            return False
        cursor = cursor[2+e]
        history += ((i,e),)
    return history == target

def hcr_query_bounds(planner, context):
    if planner.rectangular:
        raise ValueError("LB/UB checks are for HCR.")
    known = [HistoryPlanner([model],planner.rep,False) for model in planner.models]
    rows = []
    main_policy = planner.solve_state()[1]
    for history,data in list(planner.state_data.items()):
        if not data["active"]:
            continue
        for query_label, continuation in data["query_values"].items():
            if query_label == "composite":
                children = [((0,a),(1,b)) for a,b in CELLS]
                pred = planner.composite_predictive()
                active_indices = list(range(len(planner.models)))
                cost = float(QUERY_COSTS.sum())
            else:
                source = int(query_label)
                children = [history+((source,e),) for e in (0,1)]
                active, _, _, pred = posterior_and_predictive(planner.tables,history,source)
                active_indices = np.flatnonzero(active).tolist()
                cost = float(QUERY_COSTS[source])
            oracle_values = np.array([[known[k].solve_state(h)[0] for h in children] for k in active_indices])
            robust_stop_values = np.array([robust_stop(planner.beliefs(h))[0] if len(planner.beliefs(h)) else 0.0 for h in children])
            lower = float(cost + np.max(np.sum(pred*oracle_values,axis=1)))
            upper = float(cost + np.max(pred @ robust_stop_values))
            if lower > continuation+VALUE_TOL or continuation > upper+VALUE_TOL:
                raise AssertionError("HCR query LB <= continuation <= UB failed.")
            stop = float(data["stop_value"])
            reachable = (not history or (len(history)==2 and planner.rep=="composite" and not isinstance(main_policy,int))) if planner.rep == "composite" else on_policy_history(main_policy,history)
            rows.append({**context,"history":history_text(history),"query":query_label,
                "on_chosen_policy":reachable,"stop_value":stop,
                "query_lower_bound":lower,"hcr_query_value":continuation,"query_upper_bound":upper,
                "robust_improvement_lower":stop-upper,"robust_improvement_upper":stop-lower,
                "stop_certificate_lb_at_least_stop":lower >= stop-VALUE_TOL,
                "continue_certificate_ub_below_stop":upper < stop-VALUE_TOL,
                "lower_slack":continuation-lower,"upper_slack":upper-continuation})
    return rows

def planner_state_rows(hcr, rect, context):
    rows = []
    histories = sorted(set(hcr.state_data)|set(rect.state_data),key=lambda h:(len(h),h))
    for history in histories:
        hb, rb = hcr.beliefs(history), rect.beliefs(history)
        hd, rd = hcr.state_data.get(history,{}), rect.state_data.get(history,{})
        row = {**context,"history":history_text(history),"history_length":len(history),
            "hcr_active":len(hb)>0,"rect_active":len(rb)>0,
            "hcr_belief_entries":len(hb),"rect_belief_entries":len(rb)}
        if len(hb):
            row.update(hcr_posterior_y1_min=float(hb[:,1].min()), hcr_posterior_y1_max=float(hb[:,1].max()),
                hcr_stop_value=robust_stop(hb)[0], hcr_value=hcr.solve_state(history)[0],
                hcr_chosen_action=policy_json(hcr.solve_state(history)[1])["action"])
        if len(rb):
            row.update(rect_posterior_y1_min=float(rb[:,1].min()), rect_posterior_y1_max=float(rb[:,1].max()),
                rect_stop_value=robust_stop(rb)[0], rect_value=rect.solve_state(history)[0],
                rect_chosen_action=policy_json(rect.solve_state(history)[1])["action"])
        if len(hb) and len(rb):
            row["rect_stop_minus_hcr_stop"] = row["rect_stop_value"]-row["hcr_stop_value"]
        rows.append(row)
    return rows

def splicing_witness_rows(planner, context):
    if planner.rep == "composite":
        return []
    rows = []
    for first_source in (0,1):
        other = 1-first_source
        for first_bit in (0,1):
            first = ((first_source,first_bit),)
            _,b,kernel,_ = posterior_and_predictive(planner.tables,first,other)
            for a in range(len(planner.models)):
                for c in range(len(planner.models)):
                    pair_compatible = any(np.allclose(b[a],b[j],atol=PROB_TOL,rtol=0.0) and
                        np.allclose(kernel[c],kernel[j],atol=PROB_TOL,rtol=0.0) for j in range(len(planner.models)))
                    for second_bit in (0,1):
                        weight = b[a]*kernel[c,:,second_bit]
                        predictive = float(weight.sum())
                        rows.append({**context,"first_history":history_text(first),"next_source":other+1,
                            "posterior_model_id":planner.models[a]["id"], "kernel_model_id":planner.models[c]["id"],
                            "same_model_label":a==c,"pair_compatible_with_any_complete_model":pair_compatible,
                            "current_posterior_y1":float(b[a,1]),
                            "q_next1_given_y0_history":float(kernel[c,0,1]),
                            "q_next1_given_y1_history":float(kernel[c,1,1]),
                            "second_observation":second_bit,"predictive_probability":predictive,
                            "spliced_posterior_y1":float(weight[1]/predictive) if predictive>0 else None,
                            "positive_predictive":predictive>0})
    return rows

def physical_and_summary_rows(physical, nominal, planning, solutions, context):
    summaries, physical_rows, policies = [],[],[]
    for method, solution in solutions.items():
        rep = solution["representation"]
        values = mixture_metrics(physical,rep,solution)
        signature = solution_signature(solution)
        nominal_phys_index = next(j for j,m in enumerate(physical) if same_law(m,nominal))
        worst_index = int(np.argmax(values["risk"]))
        for j,truth in enumerate(physical):
            coverage = is_represented(truth,planning)
            certificate_applies = coverage if method != "Nominal" else same_law(truth,next(m for m in planning if m["id"]==solution["nominal_model_id"]))
            gap = float(values["risk"][j]-solution["objective"])
            if certificate_applies and gap>VALUE_TOL:
                raise AssertionError("A physically covered policy violates its reported risk bound.")
            physical_rows.append({**context,"method":method,"truth_model_id":truth["id"],
                "truth_profile":truth["profile"],"truth_t1":truth["t1"],"truth_t0":truth["t0"],
                "policy_signature":signature,"model_objective":solution["objective"],
                **{"true_"+key:float(value[j]) for key,value in values.items()},
                "complete_joint_law_represented":coverage,
                "marginal_profile_represented":marginal_profile_represented(truth,planning),
                "certificate_applies_to_this_truth":certificate_applies,
                "true_risk_minus_objective":gap,"reported_bound_exceeded":gap>VALUE_TOL})
        summaries.append({**context,"method":method,"policy_signature":signature,
            "planning_model_count":len(planning),"physical_library_model_count":len(physical),
            "model_objective":solution["objective"],"worst_physical_risk":float(values["risk"][worst_index]),
            "worst_physical_model_id":physical[worst_index]["id"],
            "worst_physical_wrong":float(values["wrong"][worst_index]),
            "worst_physical_abstain":float(values["abstain"][worst_index]),
            "worst_physical_original_queries":float(values["original_queries"][worst_index]),
            "worst_physical_logical_queries":float(values["logical_queries"][worst_index]),
            "central_true_risk":float(values["risk"][nominal_phys_index]),
            "central_true_wrong":float(values["wrong"][nominal_phys_index]),
            "central_true_abstain":float(values["abstain"][nominal_phys_index]),
            "central_original_queries":float(values["original_queries"][nominal_phys_index]),
            "central_logical_queries":float(values["logical_queries"][nominal_phys_index]),
            "all_physical_models_represented":all(is_represented(m,planning) for m in physical),
            "support_size":len(solution["support_indices"]),
            **{k:solution[k] for k in ("lp_primal_residual","lp_duality_gap","lp_stationarity_residual",
                "lp_complementarity_residual","static_deterministic_objective","randomization_gain") if k in solution}})
        basis = policies_for(rep)
        policies.append({**context,"method":method,"policy_signature":signature,
            "planning_model_ids":[m["id"] for m in planning],"objective":solution["objective"],
            "solver_details":{k:v for k,v in solution.items() if k not in ("support_indices","support_weights")},
            "components":[{"weight":w,"policy_id":policy_id(basis[i]),"policy":policy_json(basis[i]),"raw_policy":basis[i]}
                for i,w in zip(solution["support_indices"],solution["support_weights"])]})
    return summaries,physical_rows,policies

def policy_path_rows(solution, truth, planning, context):
    rows = []
    tables = np.stack([m["table"] for m in planning])
    true_joint = truth["table"] * np.array([1.0-PRIOR_Y1,PRIOR_Y1])[:,None,None]
    rect_planner = HistoryPlanner(planning,solution["representation"],True) if solution["method"]=="Rectangular" else None
    for policy_index, mixture_weight in zip(solution["support_indices"],solution["support_weights"]):
        policy = policies_for(solution["representation"])[policy_index]
        grouped = {}
        for cell in CELLS:
            action,queries,logical,history = traverse(policy,cell)
            key = (history,action,tuple(queries),logical)
            grouped.setdefault(key,np.zeros(2))
            grouped[key] += true_joint[:,cell[0],cell[1]]
        for (history,action,queries,logical), y_mass in grouped.items():
            probability = float(y_mass.sum())
            if probability == 0.0:
                continue
            if rect_planner is None:
                _,beliefs,_,_ = posterior_and_predictive(tables,history)
            else:
                beliefs = rect_planner.beliefs(history)
            true_posterior = float(y_mass[1]/probability)
            rows.append({**context,"method":solution["method"],"truth_model_id":truth["id"],
                "component_policy_id":policy_id(policy),"mixture_weight":mixture_weight,
                "history":history_text(history),"terminal_action":ACTION_LABELS[action],
                "path_probability_given_component":probability,
                "weighted_path_probability":mixture_weight*probability,
                "true_posterior_y1":true_posterior,"planning_history_possible":len(beliefs)>0,
                "planning_posterior_y1_min":float(beliefs[:,1].min()) if len(beliefs) else None,
                "planning_posterior_y1_max":float(beliefs[:,1].max()) if len(beliefs) else None,
                "original_queries":float(sum(queries)),"logical_queries":logical,
                "query_cost":float(np.dot(queries,QUERY_COSTS)),
                "conditional_wrong_probability":float(np.dot(y_mass/probability,LOSS[action])) if action!=0 else 0.0,
                "conditional_abstention":action==0})
    if abs(sum(row["weighted_path_probability"] for row in rows)-1.0)>VALUE_TOL:
        raise AssertionError("Physical path probabilities do not sum to one.")
    return rows

def checkpoint_path(key):
    return CHECKPOINT_DIR / (hashlib.sha256(str(key).encode()).hexdigest()+".json")

def checkpoint_read(key):
    path = checkpoint_path(key)
    if not RESUME or not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("fingerprint")!=RUN_FINGERPRINT or payload.get("key")!=key:
        raise RuntimeError("Checkpoint identity mismatch: " + str(key))
    return payload["data"]

def checkpoint_write(key, data):
    atomic_json(checkpoint_path(key),{"fingerprint":RUN_FINGERPRINT,"key":key,"data":data})

def require_ready():
    if not RUN_READY:
        raise RuntimeError("Run the analytic-controls cell successfully before the sweep.")

def analytic_controls():
    rows = []
    def check(name,observed,expected,tolerance=VALUE_TOL):
        if not np.isfinite(observed) or abs(observed-expected)>tolerance:
            raise AssertionError("{}: observed={:.15g}, expected={:.15g}".format(name,observed,expected))
        rows.append({"control":name,"observed":float(observed),"expected":float(expected),"absolute_error":abs(observed-expected),"passed":True})
    check("candidate count",len(CATALOG),300,0)
    check("unique candidate count",len(unique_models(CATALOG)),300,0)
    check("sequential policy count",len(SEQUENTIAL_POLICIES),291,0)
    check("composite policy count",len(COMPOSITE_POLICIES),84,0)
    for model in CATALOG:
        for parameter in model["parameters"]:
            y,t=parameter["y"],parameter["t"]
            q1,q2=parameter["q1"],parameter["q2"]
            if t==1.0:
                if q1<=q2:
                    check("exact Frechet upper zero10: {} / Y={}".format(model["id"],y),model["table"][y,1,0],0.0,0.0)
                if q2<=q1:
                    check("exact Frechet upper zero01: {} / Y={}".format(model["id"],y),model["table"][y,0,1],0.0,0.0)
            elif t==-1.0:
                a,b=(0,0) if q1+q2>=1.0 else (1,1)
                check("exact Frechet lower zero: {} / Y={}".format(model["id"],y),model["table"][y,a,b],0.0,0.0)
        projected = product_projection(model)
        if not same_law(projected,product_projection(projected)):
            raise AssertionError("Product projection is not idempotent.")
        for y in (0,1):
            joint=projected["table"][y]
            check("product covariance: {} / Y={}".format(model["id"],y),
                float(joint[1,1]-joint[1,:].sum()*joint[:,1].sum()),0.0,PROB_TOL)
    for t1,t0,label in ((0.0,0.0,"independent"),(1.0,1.0,"copies"),(0.25,0.0,"asymmetric")):
        truth = CATALOG_BY_ID[model_id("B",t1,t0)]
        all_solutions = {}
        for rep in REPS:
            planning,index = representation_models([truth],truth,rep)
            solutions,hp,rp = solve_methods(planning,index,rep)
            for method,sol in solutions.items():
                risk=mixture_metrics([truth],rep,sol)["risk"][0]
                all_solutions[(rep,method)] = (sol,risk)
                if label=="independent":
                    check("independent objective {}/{}".format(rep,method),sol["objective"],0.082)
                    check("independent true risk {}/{}".format(rep,method),risk,0.082)
                if label=="copies":
                    check("copies true risk {}/{}".format(rep,method),risk,0.210 if rep=="independence" else 0.100)
                    check("copies objective {}/{}".format(rep,method),sol["objective"],0.082 if rep=="independence" else 0.100)
                if label=="asymmetric":
                    check("asymmetric true risk {}/{}".format(rep,method),risk,{"independence":0.098,"joint_sequential":0.0915,"composite":0.094}[rep])
                    check("asymmetric objective {}/{}".format(rep,method),sol["objective"],{"independence":0.082,"joint_sequential":0.0915,"composite":0.094}[rep])
            hcr_query_bounds(hp,{"control":label,"representation":rep})
            if label=="asymmetric" and rep=="joint_sequential":
                values=mixture_metrics([truth],rep,solutions["HCR"])
                check("asymmetric sequential original queries",values["original_queries"][0],1.5)
                check("asymmetric first1 second-query value",hp.state_data[((0,1),)]["query_values"]["1"],0.073)
                check("asymmetric first0 second-query value",hp.state_data[((0,0),)]["query_values"]["1"],0.105)
                _,b,k,_=posterior_and_predictive(planning[0]["table"][None,:,:,:],((0,1),),1)
                check("asymmetric Q(E2=1|Y1,E1=1)",k[0,1,1],0.85)
                check("asymmetric Q(E2=1|Y0,E1=1)",k[0,0,1],0.20)
            if label=="asymmetric" and rep=="composite":
                values=mixture_metrics([truth],rep,solutions["HCR"])
                check("composite original acquisitions",values["original_queries"][0],2.0)
                check("composite logical query count",values["logical_queries"][0],1.0)
    # A nontrivial three-model library also exercises the randomized Static LP.
    physical,nominal = robust_library(0.0,"matched_size_K3")
    for rep in REPS:
        planning,index = representation_models(physical,nominal,rep)
        solutions,hp,rp = solve_methods(planning,index,rep)
        physical_and_summary_rows(physical,nominal,planning,solutions,{"control":"K3 at independence","representation":rep})
        hcr_query_bounds(hp,{"control":"K3 at independence","representation":rep})
    # State-sufficiency diagnostic: E2 = E1 XOR Y, with E1 fair in each class.
    xor_table=np.zeros((2,2,2))
    for y in (0,1):
        for e1 in (0,1):
            xor_table[y,e1,e1^y]=0.5
    _,b0,k0,_=posterior_and_predictive(xor_table[None,:,:,:],((0,0),),1)
    _,b1,k1,_=posterior_and_predictive(xor_table[None,:,:,:],((0,1),),1)
    check("XOR: first histories have identical posterior",float(np.max(np.abs(b0-b1))),0.0)
    check("XOR: next kernels still differ",float(np.max(np.abs(k0-k1))),1.0)
    save_frame("issue8_analytic_controls.csv",rows)
    atomic_json(OUTPUT_DIR / "issue8_analytic_controls.json",{"all_passed":True,"checks":len(rows),"examples":"Analytic expectations, verified by this notebook run; no pre-filled outputs."})
    print("All analytic controls passed:",len(rows))
    return rows


## Run the analytic controls first

These values are derived from the frozen synthetic law and are checked by the next cell.

| Central profile B | Independence model value | Independence policy's physical risk | Correct joint sequential optimum | Composite optimum |
|---|---:|---:|---:|---:|
| $(t_1,t_0)=(0,0)$ | .082 | .082 | .082 | .082 |
| perfect copies $(1,1)$ | .082 | .210 | .100 | .100 |
| asymmetric $(.25,0)$ | .082 | .098 | .0915 | .0940 |

For the asymmetric law, querying source1 first yields continuation values .073 after observing 1
and .105 after observing 0, against stop value .100. The optimum queries the second source only after 1;
it acquires 1.5 sources on average. Composite acquires both sources, with one logical action.
The checks also cover normalization, marginal preservation, projection idempotence, policy counts,
the randomized Static LP, within-representation hierarchy, frozen-policy HCR recursion, and query bounds.

The HCR lower query bound uses known-model optimal continuation for each candidate model; the upper
query bound commits to robust stopping immediately after that query. We verify $LB_i\le C_i\le UB_i$.
The associated improvement interval bounds $V_{\rm stop}-C_i$ in the planning model;
it is not a statement of improvement under every physical law.


In [ ]:
CONTROL_ROWS = safely_run('analytic_controls', analytic_controls)
RUN_READY = True


## Known-law sweep and robust libraries

The known-law sweep evaluates every physical law under all three representations and all four solvers.
The robust sweeps use the following two pre-specified families for every $s$ on the grid:

* **Matched size K3:** $\Theta_s^{(3)}=\{P_{r,s,s}:r=A,B,C\}$.
* **Uncertain dependence:** $D_s=\{(0,0),(s,s),(s,0),(0,s)\}$ and
  $\Theta_s^J=\{P_{r,t_1,t_0}:r=A,B,C,(t_1,t_0)\in D_s\}$.

Complete-law deduplication gives K=12 except K=3 at $s=0$ in the second family.
Both families project to three product models under independence planning.
The nominal model is profile B at $(s,s)$ and is retained after deduplication.
Each solved policy is evaluated on **every physical law in its generating library**.
Main robust physical endpoints are the worst fixed-model risk and the central truth's metrics;
no arbitrary mixture over candidate truths is imposed.
Libraries at different $s$ are **not nested**, so these curves are not ambiguity-monotonicity tests.

For Static, HCR and Rectangular, physical coverage of the complete law implies the associated fixed-law
risk bound. For Nominal that claim applies only at its nominal law. Under independence planning,
matching marginals does not establish complete-law coverage. Exceedance and applicability are separate columns.
Policy paths, posteriors, query certificates and selected complete splicing witnesses are exported.
Off-diagonal model labels alone do not prove incompatibility: witnesses also test whether the spliced pair
is realizable by **any** complete candidate model.


In [ ]:
def run_known_law_sweep():
    require_ready()
    summary_rows, physical_rows, policy_records, path_rows, state_rows, bound_rows = [],[],[],[],[],[]
    fresh = 0
    for number,truth in enumerate(CATALOG,1):
        key = "known:"+truth["id"]
        result = checkpoint_read(key)
        if result is None:
            result={"summary":[],"physical":[],"policies":[],"paths":[],"states":[],"bounds":[]}
            for rep in REPS:
                context={"experiment":"known_law","truth_model_id":truth["id"],
                    "truth_profile":truth["profile"],"truth_t1":truth["t1"],"truth_t0":truth["t0"],"representation":rep}
                planning,nominal_index=representation_models([truth],truth,rep)
                solutions,hp,rp=solve_methods(planning,nominal_index,rep)
                summaries,physical,policies=physical_and_summary_rows([truth],truth,planning,solutions,context)
                result["summary"].extend(summaries)
                result["physical"].extend(physical)
                result["policies"].extend(policies)
                for solution in solutions.values():
                    result["paths"].extend(policy_path_rows(solution,truth,planning,context))
                result["states"].extend(planner_state_rows(hp,rp,context))
                result["bounds"].extend(hcr_query_bounds(hp,context))
            checkpoint_write(key,result)
            fresh += 1
        summary_rows.extend(result["summary"])
        physical_rows.extend(result["physical"])
        policy_records.extend(result["policies"])
        path_rows.extend(result["paths"])
        state_rows.extend(result["states"])
        bound_rows.extend(result["bounds"])
        if number%25==0 or number==len(CATALOG):
            print("Known-law sweep: {}/{} laws; {} newly computed.".format(number,len(CATALOG),fresh),flush=True)
    expected=len(CATALOG)*len(REPS)*len(METHODS)
    if len(summary_rows)!=expected or len(physical_rows)!=expected:
        raise AssertionError("Known-law sweep is incomplete or duplicated.")
    frames={
        "summary":save_frame("issue8_known_law_summary.csv",summary_rows),
        "physical":save_frame("issue8_known_law_physical_risks.csv",physical_rows),
        "paths":save_frame("issue8_known_law_paths.csv",path_rows),
        "states":save_frame("issue8_known_law_states.csv",state_rows),
        "bounds":save_frame("issue8_known_law_query_bounds.csv",bound_rows),
    }
    atomic_json(OUTPUT_DIR/"issue8_known_law_policies.json",policy_records)
    return frames

def run_robust_library_sweep():
    require_ready()
    summary_rows,physical_rows,policy_records,path_rows,state_rows,bound_rows,splice_rows,library_rows=[],[],[],[],[],[],[],[]
    configurations=list(itertools.product(("matched_size_K3","uncertain_dependence"),DEPENDENCE_GRID))
    for number,(family,s) in enumerate(configurations,1):
        key="robust:{}:{:+.3f}".format(family,s)
        result=checkpoint_read(key)
        if result is None:
            physical,nominal=robust_library(s,family)
            result={"summary":[],"physical":[],"policies":[],"paths":[],"states":[],"bounds":[],"splices":[],"libraries":[]}
            for rep in REPS:
                context={"experiment":"robust_library","library_family":family,"s":float(s),"representation":rep}
                planning,nominal_index=representation_models(physical,nominal,rep)
                if rep=="independence" and len(planning)!=3:
                    raise AssertionError("Independent projections must deduplicate to three models.")
                if family=="matched_size_K3" and len(physical)!=3:
                    raise AssertionError("The matched-size library must contain exactly three models.")
                if family=="uncertain_dependence" and len(physical)!=(3 if s==0.0 else 12):
                    raise AssertionError("Unexpected uncertain-dependence library size.")
                solutions,hp,rp=solve_methods(planning,nominal_index,rep)
                summaries,phys,policies=physical_and_summary_rows(physical,nominal,planning,solutions,context)
                result["summary"].extend(summaries)
                result["physical"].extend(phys)
                result["policies"].extend(policies)
                result["states"].extend(planner_state_rows(hp,rp,context))
                result["bounds"].extend(hcr_query_bounds(hp,context))
                for solution in solutions.values():
                    for truth in physical:
                        result["paths"].extend(policy_path_rows(solution,truth,planning,context))
                for model in planning:
                    result["libraries"].append({**context,"role":"planning", "model_id":model["id"],
                        "is_nominal":same_law(model,planning[nominal_index]),"profile":model["profile"],"t1":model["t1"],"t0":model["t0"]})
                for model in physical:
                    result["libraries"].append({**context,"role":"physical_evaluation","model_id":model["id"],
                        "is_nominal":same_law(model,nominal),"profile":model["profile"],"t1":model["t1"],"t0":model["t0"]})
                # Complete splicing witnesses for two pre-specified illustrative libraries.
                if rep=="joint_sequential" and ((family=="matched_size_K3" and s==0.0) or (family=="uncertain_dependence" and s==0.75)):
                    result["splices"].extend(splicing_witness_rows(rp,context))
            checkpoint_write(key,result)
        summary_rows.extend(result["summary"])
        physical_rows.extend(result["physical"])
        policy_records.extend(result["policies"])
        path_rows.extend(result["paths"])
        state_rows.extend(result["states"])
        bound_rows.extend(result["bounds"])
        splice_rows.extend(result["splices"])
        library_rows.extend(result["libraries"])
        print("Robust sweep: {}/{} libraries; {} s={:+.3f}.".format(number,len(configurations),family,s),flush=True)
    if len(summary_rows)!=len(configurations)*len(REPS)*len(METHODS):
        raise AssertionError("Robust library sweep is incomplete or duplicated.")
    frames={"summary":save_frame("issue8_robust_summary.csv",summary_rows),
        "physical":save_frame("issue8_robust_physical_risks.csv",physical_rows),
        "paths":save_frame("issue8_robust_paths.csv",path_rows),
        "states":save_frame("issue8_robust_states.csv",state_rows),
        "bounds":save_frame("issue8_robust_query_bounds.csv",bound_rows),
        "splices":save_frame("issue8_rectangular_splicing_witnesses.csv",splice_rows),
        "libraries":save_frame("issue8_robust_library_membership.csv",library_rows)}
    atomic_json(OUTPUT_DIR/"issue8_robust_policies.json",policy_records)
    return frames


In [ ]:
KNOWN_FRAMES = safely_run('known_law_sweep', run_known_law_sweep)
ROBUST_FRAMES = safely_run('robust_library_sweep', run_robust_library_sweep)


## Paired finite-sample calibration: complete-law coverage versus marginal coverage

Truths are fixed **before sampling** at profile B with $(t_1,t_0)=(0,0),(.25,0),(.75,.75),(1,1)$,
all contained in the fixed 300-law candidate family. For each truth and each $n\in\{50,200,1000\}$,
sample n iid paired episodes per class Y (total 2n). The multinomial count draw is exactly equivalent
to iid draws from the four joint cells; marginal counts are derived from those same paired episodes.
Sampling is independent across episodes, not across sources within an episode.

* Joint calibration constructs two-sided Clopper–Pearson intervals for all eight class/cell probabilities,
  each with failure budget $.05/8$, and filters the fixed 300-law family.
* Marginal calibration constructs intervals for the four class/source success probabilities,
  each with budget $.05/4$, and filters the three product profiles.

The union bound uses no source independence: $P(\exists j\text{ failed interval})\le\sum_j\alpha_j=.05$.
Its binomial intervals require iid calibration episodes within each class.
The complete joint candidate family contains the chosen truths; coverage claims do not extend to arbitrary
continuous laws outside that family. Marginal calibration cannot certify a correlated complete law.
No truth is silently inserted into a calibrated set and no interval is silently widened.

HCR is solved on each nonempty calibrated set. The true-law risk is then computed exactly.
The notebook reports complete-law and marginal coverage, set size, physical risk, wrong answers,
abstention, query counts, and bound exceedance. Cached repeated sets preserve the same result;
fresh-solve timings and cache hits are labelled separately.
For an empty set, a **separate constant-abstain fallback** has unconditional risk/bound .10;
its HCR objective is left undefined. Overall and nonempty-HCR denominators are reported explicitly.
Wilson intervals describe Monte Carlo **rates across calibration repetitions**, not approximation error
in the exact physical-risk calculations. For a covered complete law, every HCR certificate check must pass;
empirical coverage is allowed to fluctuate and is not forced to equal or exceed 95% in every Monte Carlo run.


In [ ]:
def cp_interval_lookup(n, per_coordinate_alpha):
    counts=np.arange(n+1)
    lower=np.zeros(n+1)
    upper=np.ones(n+1)
    interior_lower=counts>0
    interior_upper=counts<n
    lower[interior_lower]=beta.ppf(per_coordinate_alpha/2.0,
        counts[interior_lower],n-counts[interior_lower]+1)
    upper[interior_upper]=beta.ppf(1.0-per_coordinate_alpha/2.0,
        counts[interior_upper]+1,n-counts[interior_upper])
    if not np.all(np.isfinite(lower)) or not np.all(np.isfinite(upper)) or np.any(lower>upper):
        raise AssertionError("Invalid Clopper-Pearson lookup.")
    return lower,upper

CP_JOINT={n:cp_interval_lookup(n,CALIBRATION_FAILURE_BUDGET/8.0) for n in CALIBRATION_N_PER_CLASS}
CP_MARGINAL={n:cp_interval_lookup(n,CALIBRATION_FAILURE_BUDGET/4.0) for n in CALIBRATION_N_PER_CLASS}
JOINT_CANDIDATE_CELLS=CATALOG_TABLES.reshape(len(CATALOG),2,4)
PRODUCT_CANDIDATE_SUCCESSES=np.stack([np.column_stack([t[:,1,:].sum(axis=1),t[:,:,1].sum(axis=1)]) for t in PRODUCT_TABLES])
CALIBRATION_SOLUTION_CACHE={}
CALIBRATION_POLICY_REGISTRY={}

def calibrated_model_indices(counts,n):
    jl,ju=CP_JOINT[n]
    ml,mu=CP_MARGINAL[n]
    joint_lower,joint_upper=jl[counts],ju[counts]
    joint_mask=np.all((JOINT_CANDIDATE_CELLS>=joint_lower[None,:,:]) &
        (JOINT_CANDIDATE_CELLS<=joint_upper[None,:,:]),axis=(1,2))
    # The marginal counts are derived from the SAME paired episodes.
    marginal_counts=np.column_stack([counts[:,2]+counts[:,3],counts[:,1]+counts[:,3]])
    marginal_lower,marginal_upper=ml[marginal_counts],mu[marginal_counts]
    marginal_mask=np.all((PRODUCT_CANDIDATE_SUCCESSES>=marginal_lower[None,:,:]) &
        (PRODUCT_CANDIDATE_SUCCESSES<=marginal_upper[None,:,:]),axis=(1,2))
    return np.flatnonzero(joint_mask),np.flatnonzero(marginal_mask),{
        "joint_lower":joint_lower.tolist(),"joint_upper":joint_upper.tolist(),
        "marginal_success_counts":marginal_counts.tolist(),
        "marginal_lower":marginal_lower.tolist(),"marginal_upper":marginal_upper.tolist()}

def cached_calibration_solution(indices,rep):
    key=(rep,tuple(int(i) for i in indices))
    if not len(indices):
        raise ValueError("An empty calibration set must use the explicitly labelled fallback.")
    if key in CALIBRATION_SOLUTION_CACHE:
        return CALIBRATION_SOLUTION_CACHE[key],True,0.0
    candidates=CATALOG if rep=="joint_sequential" else PRODUCT_CANDIDATES
    models=[candidates[int(i)] for i in indices]
    start=time.perf_counter()
    solution=solve_history(models,rep,False,False)
    elapsed=time.perf_counter()-start
    signature=solution_signature(solution)
    if signature not in CALIBRATION_POLICY_REGISTRY:
        basis=policies_for(rep)
        CALIBRATION_POLICY_REGISTRY[signature]={"policy_signature":signature,"representations":[rep],
            "policy":policy_json(basis[solution["support_indices"][0]]),
            "raw_policy":basis[solution["support_indices"][0]],"method":"HCR"}
    elif rep not in CALIBRATION_POLICY_REGISTRY[signature]["representations"]:
        CALIBRATION_POLICY_REGISTRY[signature]["representations"].append(rep)
    CALIBRATION_SOLUTION_CACHE[key]=solution
    return solution,False,elapsed

def one_calibration_repetition(truth,truth_index,n,repetition):
    rng=np.random.default_rng(np.random.SeedSequence([MASTER_SEED,truth_index,n,repetition]))
    counts=np.stack([rng.multinomial(n,truth["table"][y].reshape(4)) for y in (0,1)])
    joint_indices,product_indices,intervals=calibrated_model_indices(counts,n)
    output=[]
    true_joint_index=CATALOG_INDEX[truth["id"]]
    true_product_index=list(PROFILE_R).index(truth["profile"])
    for rep,indices in (("joint_sequential",joint_indices),("independence",product_indices)):
        candidates=CATALOG if rep=="joint_sequential" else PRODUCT_CANDIDATES
        models=[candidates[int(i)] for i in indices]
        complete_coverage=(true_joint_index in set(indices.tolist())) if rep=="joint_sequential" else (
            true_product_index in set(indices.tolist()) and same_law(truth,PRODUCT_CANDIDATES[true_product_index]))
        marginal_coverage=marginal_profile_represented(truth,models) if models else False
        base={"truth_model_id":truth["id"],"truth_index":truth_index,"truth_t1":truth["t1"],"truth_t0":truth["t0"],
            "n_per_class":n,"total_paired_episodes":2*n,"repetition":repetition,"representation":rep,
            "candidate_count":len(candidates),"calibrated_set_size":len(indices),
            "complete_joint_law_covered":bool(complete_coverage),"marginal_profile_covered":marginal_coverage,
            "set_empty":len(indices)==0,"family_failure_budget":CALIBRATION_FAILURE_BUDGET,
            "calibrated_set_hash":hashlib.sha256(canonical_json(indices.tolist()).encode()).hexdigest()[:20],
            "joint_y0_counts_json":canonical_json(counts[0].tolist()),"joint_y1_counts_json":canonical_json(counts[1].tolist())}
        if len(indices)==0:
            # No HCR optimum is defined here. Abstain is a separate safe policy.
            solution=deterministic_solution(0,ABSTENTION_COST,rep,"safe_abstain_fallback")
            bound=ABSTENTION_COST
            hcr_value=None
            cached=False
            elapsed=0.0
            solve_status="empty_set_safe_abstain_fallback"
        else:
            solution,cached,elapsed=cached_calibration_solution(indices,rep)
            bound=solution["objective"]
            hcr_value=bound
            solve_status="nonempty_HCR_solve"
        values=mixture_metrics([truth],rep,solution)
        risk=float(values["risk"][0])
        gap=risk-bound
        if len(indices)>0 and complete_coverage and gap>VALUE_TOL:
            raise AssertionError("Covered joint law violates the calibrated HCR bound.")
        if len(indices)==0 and abs(risk-ABSTENTION_COST)>VALUE_TOL:
            raise AssertionError("The safe fallback must be constant abstention.")
        output.append({**base,"solve_status":solve_status,
            "hcr_model_objective":hcr_value,"reported_policy_bound":bound,
            "fallback_safe_bound":ABSTENTION_COST if len(indices)==0 else None,
            "bound_type":"HCR_under_model_coverage" if len(indices)>0 else "constant_abstain_unconditional",
            "policy_signature":solution_signature(solution),
            **{"true_"+key:float(value[0]) for key,value in values.items()},
            "true_risk_minus_reported_bound":gap,"reported_bound_exceeded":gap>VALUE_TOL,
            "hcr_bound_exceeded":(gap>VALUE_TOL) if len(indices)>0 else None,
            "covered_HCR_certificate_violation":bool(len(indices)>0 and complete_coverage and gap>VALUE_TOL),
            "HCR_dynamic_state_count":solution.get("dynamic_state_count",0) if len(indices)>0 else 0,
            "HCR_model_table_array_bytes":solution.get("table_array_bytes",0) if len(indices)>0 else 0,
            "solver_cache_hit":cached,"fresh_HCR_solve_wall_seconds":elapsed})
    witness={"truth_model_id":truth["id"],"n_per_class":n,"repetition":repetition,
        "paired_counts_by_y":counts.tolist(),"selected_joint_model_ids":[CATALOG[int(i)]["id"] for i in joint_indices],
        "selected_product_model_ids":[PRODUCT_CANDIDATES[int(i)]["id"] for i in product_indices],**intervals}
    return output,witness

def run_finite_sample_calibration():
    require_ready()
    cp_checks=[]
    for n in CALIBRATION_N_PER_CLASS:
        for coordinates,lookup in ((8,CP_JOINT[n]),(4,CP_MARGINAL[n])):
            alpha=CALIBRATION_FAILURE_BUDGET/coordinates
            lower,upper=lookup
            expected_upper=1.0-(alpha/2.0)**(1.0/n)
            expected_lower=(alpha/2.0)**(1.0/n)
            if max(abs(upper[0]-expected_upper),abs(lower[n]-expected_lower),abs(lower[0]),abs(upper[n]-1.0))>PROB_TOL:
                raise AssertionError("CP endpoint identity failed.")
            cp_checks.append({"n_per_class":n,"bonferroni_coordinates":coordinates,
                "per_coordinate_alpha":alpha,"zero_count_upper":upper[0],"all_count_lower":lower[n],"passed":True})
    save_frame("issue8_calibration_interval_controls.csv",cp_checks)
    all_rows=[]
    witnesses=[]
    truth_models=[CATALOG_BY_ID[model_id("B",t1,t0)] for t1,t0 in CALIBRATION_TRUTHS]
    for truth_index,truth in enumerate(truth_models):
        for n in CALIBRATION_N_PER_CLASS:
            key="calibration:{}:n={}".format(truth["id"],n)
            saved=checkpoint_read(key)
            rows=[] if saved is None else saved["rows"]
            partial_witnesses=[] if saved is None else saved["witnesses"]
            # Exactly two rows per finished repetition, one for each representation.
            if len(rows)%2:
                raise RuntimeError("A calibration checkpoint contains an incomplete paired repetition.")
            completed=len(rows)//2
            if completed>CALIBRATION_REPETITIONS:
                raise RuntimeError("Checkpoint repetition count exceeds this profile.")
            if rows:
                pairs=[(r["repetition"],r["representation"]) for r in rows]
                expected_pairs=[(r,rep) for r in range(completed) for rep in ("joint_sequential","independence")]
                if pairs!=expected_pairs:
                    raise RuntimeError("Calibration checkpoint has missing or duplicated repetitions.")
            for repetition in range(completed,CALIBRATION_REPETITIONS):
                new_rows,witness=one_calibration_repetition(truth,truth_index,n,repetition)
                rows.extend(new_rows)
                # Pre-specified repetitions retained as inspectable CI/set witnesses.
                if repetition in (0,1,2,9,CALIBRATION_REPETITIONS-1):
                    partial_witnesses.append(witness)
                if (repetition+1)%25==0 or repetition+1==CALIBRATION_REPETITIONS:
                    checkpoint_write(key,{"rows":rows,"witnesses":partial_witnesses})
                    print("Calibration {} n/class={}: {}/{} paired repetitions.".format(truth["id"],n,repetition+1,CALIBRATION_REPETITIONS),flush=True)
            if completed==CALIBRATION_REPETITIONS:
                print("Calibration restored:",truth["id"],"n/class=",n,"repetitions=",completed,flush=True)
            all_rows.extend(rows)
            witnesses.extend(partial_witnesses)
    expected=4*len(CALIBRATION_N_PER_CLASS)*CALIBRATION_REPETITIONS*2
    if len(all_rows)!=expected:
        raise AssertionError("Finite-sample study is incomplete.")
    frame=save_frame("issue8_calibration_repetitions.csv",all_rows)
    atomic_json(OUTPUT_DIR/"issue8_calibration_interval_witnesses.json",witnesses)
    # Reconstruct policy records on a fully resumed run as well.
    registry=dict(CALIBRATION_POLICY_REGISTRY)
    needed_signatures=set(frame.loc[~frame["set_empty"],"policy_signature"])
    for rep in ("joint_sequential","independence"):
        for policy in SEQUENTIAL_POLICIES:
            sol=deterministic_solution(policy,0.0,rep,"HCR")
            sig=solution_signature(sol)
            if sig in needed_signatures and sig not in registry:
                registry[sig]={"policy_signature":sig,"representations":[rep],"policy":policy_json(policy),"raw_policy":policy,"method":"HCR"}
            if sig in needed_signatures and sig in registry:
                used_reps=set(frame.loc[frame["policy_signature"]==sig,"representation"])
                registry[sig]["representations"]=sorted(used_reps)
    if not needed_signatures.issubset(set(registry)):
        raise AssertionError("Some calibrated policies could not be reconstructed.")
    atomic_json(OUTPUT_DIR/"issue8_calibration_policies.json",list(registry.values()))
    return frame


In [ ]:
CALIBRATION_FRAME = safely_run('finite_sample_calibration', run_finite_sample_calibration)


## Final audits, computational accounting and review outputs

Fresh timing repetitions rebuild the model representation, then solve each method without the calibration
solution cache. Reported wall times exclude audit/path exports, file writes and plotting; the enumerated
policy bank is shared and precompiled. Construction and optimization times are reported separately.
All repetitions, medians and quartiles are exported. These are two-source measurements, not a new
large-source scalability study, and no unmeasured runtime estimate is pre-filled.

For L classes, g sources and alphabet size A, the unrestricted numbers of free channel parameters are
$Lg(A-1)$ under independence versus $L(A^g-1)$ for a dense joint channel.
Here L=g=A=2 gives **4 versus 6 free parameters**. Dense source-channel and dense joint-channel tables
both contain eight probabilities in this particular case. The implementation stores full joint tables
for all planners; actual buffer bytes, possible compact parameter counts, and Static LP matrix dimensions
are listed separately. They are specified NumPy buffer sizes, **not total process RSS or peak RAM**.
Composite trades two sequential actions for one vector action while retaining two source acquisitions
and eliminating early stopping; joint/composite evaluation also requires paired calibration data.

Figures are written as PNGs, with their source CSVs. They show modeled-bound gaps on physical laws,
worst fixed-model physical risks, and bound-exceedance rates across calibration repetitions.
The .05 dashed line in the last plot is the joint calibration failure budget; it is not an independence-model
guarantee, nor a universal robust-value ordering across representations.
This notebook writes no TeX. Text revisions belong to the next stages after the outputs are reviewed.

Implementation references (primary documentation):

* [SciPy `linprog`, HiGHS and optimization-result marginals](https://docs.scipy.org/doc/scipy/reference/optimize.linprog-highs.html)
* [SciPy exact binomial confidence intervals](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats._result_classes.BinomTestResult.proportion_ci.html)
* [SciPy Beta quantile function](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html)


In [ ]:
def wilson_interval(successes,total,alpha=0.05):
    if total==0:
        return None,None
    z=float(norm.ppf(1.0-alpha/2.0))
    p=float(successes)/total
    denominator=1.0+z*z/total
    center=(p+z*z/(2.0*total))/denominator
    radius=z*math.sqrt(p*(1.0-p)/total+z*z/(4.0*total*total))/denominator
    return max(0.0,center-radius),min(1.0,center+radius)

def add_rate(row,name,values):
    values=list(values)
    total=len(values)
    successes=int(sum(bool(v) for v in values))
    lo,hi=wilson_interval(successes,total)
    row.update({name+"_numerator":successes,name+"_denominator":total,
        name+"_rate":successes/total if total else None,
        name+"_wilson95_lower":lo,name+"_wilson95_upper":hi})

def summarize_calibration(frame):
    rows=[]
    keys=["truth_model_id","truth_t1","truth_t0","n_per_class","representation"]
    for values,group in frame.groupby(keys,sort=True):
        row=dict(zip(keys,values))
        nonempty=group.loc[~group["set_empty"]]
        covered=nonempty.loc[nonempty["complete_joint_law_covered"]]
        row.update(repetitions=len(group),nonempty_HCR_solves=len(nonempty),
            mean_calibrated_set_size=float(group["calibrated_set_size"].mean()),
            median_calibrated_set_size=float(group["calibrated_set_size"].median()),
            min_calibrated_set_size=int(group["calibrated_set_size"].min()),
            max_calibrated_set_size=int(group["calibrated_set_size"].max()),
            mean_true_risk=float(group["true_risk"].mean()),
            risk_variation_across_calibration_repetitions_se=float(group["true_risk"].std(ddof=1)/math.sqrt(len(group))),
            mean_true_wrong=float(group["true_wrong"].mean()),
            mean_true_abstain=float(group["true_abstain"].mean()),
            mean_original_queries=float(group["true_original_queries"].mean()),
            mean_logical_queries=float(group["true_logical_queries"].mean()),
            mean_HCR_objective_nonempty=float(nonempty["hcr_model_objective"].mean()) if len(nonempty) else None,
            mean_HCR_certificate_gap_nonempty=float(nonempty["true_risk_minus_reported_bound"].mean()) if len(nonempty) else None,
            max_HCR_certificate_gap_nonempty=float(nonempty["true_risk_minus_reported_bound"].max()) if len(nonempty) else None,
            max_covered_HCR_certificate_gap=float(covered["true_risk_minus_reported_bound"].max()) if len(covered) else None)
        add_rate(row,"empty_set",group["set_empty"])
        add_rate(row,"complete_joint_coverage",group["complete_joint_law_covered"])
        add_rate(row,"marginal_profile_coverage",group["marginal_profile_covered"])
        add_rate(row,"reported_bound_exceeded_all_policies",group["reported_bound_exceeded"])
        add_rate(row,"HCR_bound_exceeded_nonempty",nonempty["reported_bound_exceeded"])
        add_rate(row,"HCR_bound_exceeded_given_complete_coverage",covered["reported_bound_exceeded"])
        rows.append(row)
    return save_frame("issue8_calibration_summary.csv",rows)

def representation_comparisons(known,robust):
    representative=known.loc[known["method"]=="HCR"]
    rows=[]
    keys=["truth_model_id","truth_profile","truth_t1","truth_t0"]
    for values,group in representative.groupby(keys,sort=True):
        indexed=group.set_index("representation")
        row=dict(zip(keys,values))
        for rep in REPS:
            for metric in ("true_risk","true_wrong","true_abstain","true_original_queries","true_logical_queries","model_objective","policy_signature","reported_bound_exceeded"):
                row[rep+"_"+metric]=indexed.loc[rep,metric]
        row["joint_minus_independence_true_risk"]=float(indexed.loc["joint_sequential","true_risk"]-indexed.loc["independence","true_risk"])
        row["composite_minus_joint_true_risk"]=float(indexed.loc["composite","true_risk"]-indexed.loc["joint_sequential","true_risk"])
        row["independence_underestimation_gap"]=float(indexed.loc["independence","true_risk"]-indexed.loc["independence","model_objective"])
        row["independence_joint_same_policy"]=(indexed.loc["independence","policy_signature"]==indexed.loc["joint_sequential","policy_signature"])
        # For known laws, correct sequential Bayes is optimal in the common class;
        # it can emulate the composite action by taking both queries without stopping.
        if row["joint_minus_independence_true_risk"]>VALUE_TOL or row["composite_minus_joint_true_risk"] < -VALUE_TOL:
            raise AssertionError("Known-law Bayes policy-class inclusion check failed.")
        rows.append(row)
    known_comparison=save_frame("issue8_known_law_representation_comparison.csv",rows)
    robust_rows=[]
    for values,group in robust.groupby(["library_family","s","method"],sort=True):
        indexed=group.set_index("representation")
        row=dict(zip(("library_family","s","method"),values))
        for rep in REPS:
            for metric in ("model_objective","worst_physical_risk","central_true_risk","central_true_wrong","central_true_abstain",
                "central_original_queries","central_logical_queries","planning_model_count","policy_signature"):
                row[rep+"_"+metric]=indexed.loc[rep,metric]
        row["joint_minus_independence_worst_physical_risk"]=float(indexed.loc["joint_sequential","worst_physical_risk"]-indexed.loc["independence","worst_physical_risk"])
        row["composite_minus_joint_worst_physical_risk"]=float(indexed.loc["composite","worst_physical_risk"]-indexed.loc["joint_sequential","worst_physical_risk"])
        row["independence_joint_same_policy"]=(indexed.loc["independence","policy_signature"]==indexed.loc["joint_sequential","policy_signature"])
        # Deliberately no cross-representation robust-value ordering assertion.
        robust_rows.append(row)
    return known_comparison,save_frame("issue8_robust_representation_comparison.csv",robust_rows)

def hierarchy_audit(known,robust):
    rows=[]
    for frame,keys in ((known,["truth_model_id","representation"]),(robust,["library_family","s","representation"])):
        for values,group in frame.groupby(keys,sort=True):
            indexed=group.set_index("method")
            if len(indexed)!=4 or set(indexed.index)!=set(METHODS):
                raise AssertionError("Missing or duplicated method in hierarchy audit.")
            ordered=[float(indexed.loc[m,"model_objective"]) for m in METHODS]
            gaps=[ordered[j+1]-ordered[j] for j in range(3)]
            if min(gaps)<-VALUE_TOL:
                raise AssertionError("Final hierarchy audit failed.")
            rows.append({"experiment":group.iloc[0]["experiment"],**dict(zip(keys,values)),
                **{m+"_objective":v for m,v in zip(METHODS,ordered)},
                "Static_minus_Nominal":gaps[0],"HCR_minus_Static":gaps[1],"Rect_minus_HCR":gaps[2],
                "hierarchy_passed":True,"hierarchy_scope":"within this representation and policy class"})
    return save_frame("issue8_hierarchy_audit.csv",rows)

def run_timing_and_storage_audit():
    raw_rows=[]
    storage_rows=[]
    for family,s in itertools.product(("matched_size_K3","uncertain_dependence"),DEPENDENCE_GRID):
        original,nominal=robust_library(s,family)
        specifications=[(m["profile"],m["t1"],m["t0"]) for m in original]
        for rep in REPS:
            for repetition in range(TIMING_REPETITIONS):
                start=time.perf_counter()
                physical=[make_model(*spec) for spec in specifications]
                planning,index=representation_models(physical,nominal,rep)
                build_elapsed=time.perf_counter()-start
                k=len(planning)
                p=len(policies_for(rep))
                for method in METHODS:
                    start=time.perf_counter()
                    if method=="Nominal":
                        solution=solve_nominal(planning,index,rep)
                    elif method=="Static_randomized":
                        solution=solve_static(planning,rep)
                    else:
                        solution=solve_history(planning,rep,method=="Rectangular",False)
                    solve_elapsed=time.perf_counter()-start
                    raw_rows.append({"library_family":family,"s":float(s),"representation":rep,"method":method,
                        "timing_repetition":repetition,"model_count":k,"deterministic_policy_count":p,
                        "model_construction_wall_seconds":build_elapsed,"solve_wall_seconds":solve_elapsed,
                        "objective":solution["objective"],"dynamic_state_count":solution.get("dynamic_state_count",0),
                        "timing_scope":"fresh solve; shared precompiled policy bank; excludes audits, output writes and plots"})
            storage_rows.append({"library_family":family,"s":float(s),"representation":rep,"model_count":k,
                "free_channel_parameters_per_model":4 if rep=="independence" else 6,
                "parameter_count_scope":"unrestricted class-conditional channels; the frozen finite family additionally constrains margins and profiles",
                "dense_source_channel_probability_entries_per_model":8,
                "dense_joint_channel_probability_entries_per_model":8,
                "actual_solver_joint_table_entries":k*8,"actual_solver_joint_table_array_bytes":k*8*np.dtype(float).itemsize,
                "compact_parameter_entries_if_stored":k*(4 if rep=="independence" else 6),
                "static_risk_matrix_entries":k*p,"static_risk_matrix_array_bytes":k*p*np.dtype(float).itemsize,
                "static_inequality_matrix_array_bytes":k*(p+1)*np.dtype(float).itemsize,
                "static_lp_weight_variables":p,"static_lp_total_variables":p+1,
                "memory_measure_scope":"specified dense NumPy buffers; not process RSS or total peak memory",
                "independent_parameter_formula":"L*g*(A-1)","joint_parameter_formula":"L*(A**g-1)",
                "current_L":2,"current_g":2,"current_A":2})
            print("Timing:",family,"s=",s,"representation=",rep,flush=True)
    raw=save_frame("issue8_timing_raw.csv",raw_rows)
    summary=[]
    for values,group in raw.groupby(["library_family","s","representation","method"],sort=True):
        times=group["solve_wall_seconds"].to_numpy()
        summary.append({**dict(zip(("library_family","s","representation","method"),values)),
            "model_count":int(group["model_count"].iloc[0]),"repetitions":len(group),
            "solve_median_seconds":float(np.median(times)),"solve_min_seconds":float(np.min(times)),
            "solve_q25_seconds":float(np.quantile(times,0.25)),"solve_q75_seconds":float(np.quantile(times,0.75)),
            "model_construction_median_seconds":float(group["model_construction_wall_seconds"].median()),
            "dynamic_state_count":int(group["dynamic_state_count"].max()),
            "shared_policy_bank_precompiled":True,"complete_process_memory_measured":False})
    return raw,save_frame("issue8_timing_summary.csv",summary),save_frame("issue8_storage_accounting.csv",storage_rows)

def save_figures(known_comparison,robust_summary,calibration_summary):
    figure_paths=[]
    fig,axes=plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
    for ax,profile in zip(axes,PROFILE_R):
        group=known_comparison.loc[known_comparison["truth_profile"]==profile]
        pivot=group.pivot(index="truth_t0",columns="truth_t1",values="independence_underestimation_gap")
        matrix=pivot.reindex(index=DEPENDENCE_GRID,columns=DEPENDENCE_GRID).to_numpy()
        scale=max(float(np.abs(known_comparison["independence_underestimation_gap"]).max()),1e-6)
        im=ax.imshow(matrix,origin="lower",cmap="coolwarm",vmin=-scale,vmax=scale,aspect="auto")
        ax.set_xticks(range(len(DEPENDENCE_GRID)))
        ax.set_xticklabels([str(t) for t in DEPENDENCE_GRID],rotation=60,fontsize=8)
        ax.set_yticks(range(len(DEPENDENCE_GRID)))
        ax.set_yticklabels([str(t) for t in DEPENDENCE_GRID],fontsize=8)
        ax.set(title="Profile "+profile,xlabel="t1 (normalized dependence)",ylabel="t0 (normalized dependence)")
    fig.colorbar(im,ax=axes,label="True risk - independence model value")
    path=OUTPUT_DIR/"issue8_known_law_certificate_gap_heatmaps.png"
    fig.savefig(path,dpi=180)
    plt.show()
    plt.close(fig)
    figure_paths.append(path)
    fig,axes=plt.subplots(2,3,figsize=(13,7),constrained_layout=True,sharex=True)
    for row,family in enumerate(("matched_size_K3","uncertain_dependence")):
        for column,rep in enumerate(REPS):
            ax=axes[row,column]
            group=robust_summary.loc[(robust_summary["library_family"]==family)&(robust_summary["representation"]==rep)]
            for method in METHODS:
                data=group.loc[group["method"]==method].sort_values("s")
                ax.plot(data["s"],data["worst_physical_risk"],marker="o",markersize=3,label=method)
            ax.set(title=family+" / "+rep,xlabel="s (libraries are not nested)",ylabel="Worst fixed-model physical risk")
            ax.grid(alpha=0.25)
    axes[0,0].legend(fontsize=8)
    path=OUTPUT_DIR/"issue8_robust_physical_risk_curves.png"
    fig.savefig(path,dpi=180)
    plt.show()
    plt.close(fig)
    figure_paths.append(path)
    fig,axes=plt.subplots(1,4,figsize=(14,4),constrained_layout=True,sharey=True)
    for ax,(t1,t0) in zip(axes,CALIBRATION_TRUTHS):
        for rep in ("joint_sequential","independence"):
            data=calibration_summary.loc[(calibration_summary["truth_t1"]==t1)&(calibration_summary["truth_t0"]==t0)&(calibration_summary["representation"]==rep)].sort_values("n_per_class")
            y=data["reported_bound_exceeded_all_policies_rate"].to_numpy(dtype=float)
            lo=data["reported_bound_exceeded_all_policies_wilson95_lower"].to_numpy(dtype=float)
            hi=data["reported_bound_exceeded_all_policies_wilson95_upper"].to_numpy(dtype=float)
            ax.errorbar(data["n_per_class"],y,yerr=np.stack([np.maximum(0,y-lo),np.maximum(0,hi-y)]),marker="o",label=rep,capsize=3)
        ax.axhline(CALIBRATION_FAILURE_BUDGET,color="gray",linestyle="--",linewidth=1,label="Joint calibration failure budget")
        ax.set(title="B: (t1,t0)=({}, {})".format(t1,t0),xlabel="Paired episodes per class",ylabel="Reported bound exceedance, all repetitions")
        ax.set_xscale("log")
        ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    path=OUTPUT_DIR/"issue8_calibration_bound_exceedance.png"
    fig.savefig(path,dpi=180)
    plt.show()
    plt.close(fig)
    figure_paths.append(path)
    return figure_paths

def final_reports():
    require_ready()
    known=KNOWN_FRAMES["physical"]
    robust=ROBUST_FRAMES["summary"]
    calibrated=CALIBRATION_FRAME
    cal_summary=summarize_calibration(calibrated)
    known_comparison,robust_comparison=representation_comparisons(known,robust)
    hierarchy=hierarchy_audit(KNOWN_FRAMES["summary"],robust)
    physical=pd.concat([known,ROBUST_FRAMES["physical"]],ignore_index=True)
    applicable=physical.loc[physical["certificate_applies_to_this_truth"]]
    valid_max=float(applicable["true_risk_minus_objective"].max())
    covered_cal=calibrated.loc[(~calibrated["set_empty"])&calibrated["complete_joint_law_covered"]]
    covered_cal_max=float(covered_cal["true_risk_minus_reported_bound"].max()) if len(covered_cal) else None
    if valid_max>VALUE_TOL or (covered_cal_max is not None and covered_cal_max>VALUE_TOL):
        raise AssertionError("Final coverage-conditional certificate audit failed.")
    bounds=pd.concat([KNOWN_FRAMES["bounds"],ROBUST_FRAMES["bounds"]],ignore_index=True)
    if (bounds["lower_slack"] < -VALUE_TOL).any() or (bounds["upper_slack"] < -VALUE_TOL).any():
        raise AssertionError("Final HCR query-bound audit failed.")
    timing_raw,timing_summary,storage=run_timing_and_storage_audit()
    figure_paths=save_figures(known_comparison,robust,cal_summary)
    rounding=save_frame("issue8_roundoff_audit.csv",pd.DataFrame(ROUNDING_AUDIT,
        columns=["model","y","operation","minimum_before","amount"]))
    audit={"all_required_checks_passed":True,"profile":RUN_PROFILE,"fingerprint":RUN_FINGERPRINT,
        "known_law_rows":len(known),"known_physical_laws":len(CATALOG),
        "robust_summary_rows":len(robust),"robust_physical_rows":len(ROBUST_FRAMES["physical"]),
        "calibration_rows":len(calibrated),"calibration_repetitions_per_truth_n":CALIBRATION_REPETITIONS,
        "hierarchy_instances_checked":len(hierarchy),"HCR_query_bounds_checked":len(bounds),
        "covered_physical_certificate_instances":len(applicable),"maximum_covered_physical_gap":valid_max,
        "covered_calibrated_HCR_instances":len(covered_cal),"maximum_covered_calibrated_HCR_gap":covered_cal_max,
        "marginal_coverage_is_not_joint_coverage":True,
        "risk_evaluation":"exact finite-state expectation; not Monte Carlo",
        "randomness":"iid paired calibration episodes, sampled through equivalent multinomial cell counts",
        "empty_sets":"separate constant-abstain fallback, never reported as an HCR optimum",
        "timing_repetitions_per_configuration_method":TIMING_REPETITIONS,
        "memory_scope":"specified NumPy buffers; total process RSS and peak RAM not measured",
        "roundoff_operations_logged":len(rounding),
        "figures":[p.name for p in figure_paths],
        "no_empirical_backends_or_LLM_used":True,"no_TeX_files_modified":True,
        "completed_utc":utc_now()}
    atomic_json(OUTPUT_DIR/"issue8_validation_report.json",audit)
    main_names=["issue8_manifest.json","issue8_validation_report.json","issue8_analytic_controls.csv","issue8_calibration_interval_controls.csv",
        "issue8_known_law_representation_comparison.csv","issue8_robust_summary.csv",
        "issue8_robust_physical_risks.csv","issue8_robust_representation_comparison.csv",
        "issue8_calibration_summary.csv","issue8_calibration_repetitions.csv",
        "issue8_timing_summary.csv","issue8_storage_accounting.csv","issue8_hierarchy_audit.csv",
        "issue8_robust_query_bounds.csv","issue8_robust_policies.json"]
    atomic_json(OUTPUT_DIR/"issue8_files_to_send.json",{"send_first":main_names,
        "send_if_detailed_audit_needed":["issue8_known_law_paths.csv","issue8_robust_paths.csv","issue8_robust_states.csv",
            "issue8_rectangular_splicing_witnesses.csv","issue8_calibration_interval_witnesses.json",
            "issue8_calibration_policies.json","issue8_roundoff_audit.csv","issue8_timing_raw.csv"],
        "checkpoints_need_not_be_sent":True})
    status("all_stages","completed",all_required_checks_passed=True)
    bundle_files=main_names+["issue8_files_to_send.json","issue8_run_status.json",
        "issue8_rectangular_splicing_witnesses.csv","issue8_robust_states.csv",
        "issue8_calibration_interval_witnesses.json","issue8_calibration_policies.json",
        "issue8_distribution_catalog.csv","issue8_analytic_controls.json",
        "issue8_known_law_summary.csv","issue8_known_law_physical_risks.csv",
        "issue8_known_law_paths.csv","issue8_known_law_states.csv","issue8_known_law_query_bounds.csv",
        "issue8_known_law_policies.json","issue8_robust_paths.csv","issue8_robust_library_membership.csv",
        "issue8_timing_raw.csv","issue8_roundoff_audit.csv"]+[p.name for p in figure_paths]
    bundle=OUTPUT_DIR/"issue8_review_bundle.zip"
    with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED) as archive:
        for name in bundle_files:
            source=OUTPUT_DIR/name
            if not source.is_file():
                raise RuntimeError("Missing required bundle file: "+name)
            archive.write(source,arcname=name)
    print("\nCOMPLETE:",RUN_PROFILE)
    print("All model/hierarchy/coverage-conditional certificate checks passed.")
    print("Outputs:",OUTPUT_DIR)
    print("Start review with issue8_validation_report.json and issue8_files_to_send.json.")
    print("Send for review:",bundle.name)
    print("Final paper tables should use FINAL outputs, not the pilot.")
    display(cal_summary[["truth_t1","truth_t0","n_per_class","representation","mean_true_risk",
        "complete_joint_coverage_rate","marginal_profile_coverage_rate","HCR_bound_exceeded_nonempty_rate","empty_set_rate"]])
    return audit


In [ ]:
FINAL_AUDIT = safely_run('final_reporting', final_reports)


## Completion and outputs

A completed run prints its profile/status and creates `issue8_review_bundle.zip` in the configured output directory. The bundle contains numerical results and audits; `issue8_run_status.json` records failures. Checkpoints support resumption under the same configuration.
